In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("/content/sample_data/clean_retention_features_updated_new.csv")

In [ ]:
features = [
    "num__DistanceFromHome",
    "num__JobSatisfaction",
    "num__EnvironmentSatisfaction",
    "num__WorkLifeBalance",
    "num__RelationshipSatisfaction",
    "num__JobInvolvement",
    "num__TrainingTimesLastYear",
    "num__YearsSinceLastPromotion",
    "num__YearsInCurrentRole",
    "num__TotalWorkingYears",
    "num__JobLevel",
    "num__MonthlyIncome",
    "num__PercentSalaryHike",
    "num__PerformanceRating",
    "num__YearsWithCurrManager",
    "cat__Department_Cardiology",
    "cat__Department_Maternity",
    "cat__Department_Neurology",
    "cat__JobRoleGroup_Clinical",
    "OverTime"
]


In [ ]:
silhouette_scores = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(df[features])
    score = silhouette_score(df[features], labels)
    silhouette_scores[k] = score

silhouette_scores


{2: np.float64(0.2541565646854647),
 3: np.float64(0.2324972070750563),
 4: np.float64(0.24062260655486553),
 5: np.float64(0.2436115762146545),
 6: np.float64(0.19969270131630423)}

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
df["Cluster"] = kmeans.fit_predict(df[features])


In [ ]:
cluster_profile = df.groupby("Cluster")[features].mean()
cluster_profile


,num__DistanceFromHome,num__JobSatisfaction,num__EnvironmentSatisfaction,num__WorkLifeBalance,num__RelationshipSatisfaction,num__JobInvolvement,num__TrainingTimesLastYear,num__YearsSinceLastPromotion,num__YearsInCurrentRole,num__TotalWorkingYears,num__JobLevel,num__MonthlyIncome,num__PercentSalaryHike,num__PerformanceRating,num__YearsWithCurrManager,cat__Department_Cardiology,cat__Department_Maternity,cat__Department_Neurology,cat__JobRoleGroup_Clinical,OverTime
Cluster,,,,,,,,,,,,,,,,,,,,
0,0.225411,0.576373,0.573363,0.611738,0.586155,0.569601,0.454853,0.140707,0.245172,0.329938,0.375282,0.380272,0.314092,0.0,0.256739,0.392777,0.0,0.0,0.0,1.0
1,0.317179,0.588863,0.560641,0.601068,0.563692,0.572845,0.465484,0.140046,0.255784,0.265932,0.291762,0.283427,0.278359,0.0,0.260398,0.816934,0.0,0.0,1.0,0.0
2,0.306215,0.580952,0.573160,0.567100,0.570563,0.563636,0.455411,0.111342,0.205844,0.239968,0.153247,0.187557,0.300742,0.0,0.214057,0.000000,1.0,0.0,1.0,0.0
3,0.330379,0.571776,0.579886,0.570965,0.570154,0.593674,0.435118,0.120438,0.232022,0.274513,0.229319,0.264518,0.306222,0.0,0.234507,0.000000,1.0,0.0,0.0,1.0


In [ ]:
def level(value):
    if value < 0.33:
        return "Low"
    elif value < 0.66:
        return "Medium"
    else:
        return "High"


In [ ]:
cluster_interpretation = {}

for cluster_id, row in cluster_profile.iterrows():
    interpretation = {
        "Job Satisfaction": level(row["num__JobSatisfaction"]),
        "Work-Life Balance": level(row["num__WorkLifeBalance"]),
        "Monthly Income": level(row["num__MonthlyIncome"]),
        "Training": level(row["num__TrainingTimesLastYear"]),
        "Career Progression": level(row["num__YearsSinceLastPromotion"]),
        "OverTime": "High" if row["OverTime"] > 0.5 else "Low",
        "Distance Stress": level(row["num__DistanceFromHome"])
    }
    cluster_interpretation[cluster_id] = interpretation

cluster_interpretation


{0: {'Job Satisfaction': 'Medium',
  'Work-Life Balance': 'Medium',
  'Monthly Income': 'Medium',
  'Training': 'Medium',
  'Career Progression': 'Low',
  'OverTime': 'High',
  'Distance Stress': 'Low'},
 1: {'Job Satisfaction': 'Medium',
  'Work-Life Balance': 'Medium',
  'Monthly Income': 'Low',
  'Training': 'Medium',
  'Career Progression': 'Low',
  'OverTime': 'Low',
  'Distance Stress': 'Low'},
 2: {'Job Satisfaction': 'Medium',
  'Work-Life Balance': 'Medium',
  'Monthly Income': 'Low',
  'Training': 'Medium',
  'Career Progression': 'Low',
  'OverTime': 'Low',
  'Distance Stress': 'Low'},
 3: {'Job Satisfaction': 'Medium',
  'Work-Life Balance': 'Medium',
  'Monthly Income': 'Low',
  'Training': 'Medium',
  'Career Progression': 'Low',
  'OverTime': 'High',
  'Distance Stress': 'Medium'}}

In [ ]:
def assign_strategy(cluster_info):
    strategies = []

    if cluster_info["Monthly Income"] == "Low":
        strategies.append("Compensation review and financial incentives")

    if cluster_info["Work-Life Balance"] == "Low" or cluster_info["OverTime"] == "High":
        strategies.append("Flexible working hours and workload management")

    if cluster_info["Career Progression"] == "High":
        strategies.append("Career advancement and promotion planning")

    if cluster_info["Training"] == "Low":
        strategies.append("Training and upskilling programs")

    if cluster_info["Job Satisfaction"] == "Low":
        strategies.append("Employee engagement and counseling programs")

    if cluster_info["Distance Stress"] == "High":
        strategies.append("Hybrid or remote work options")

    return strategies


In [ ]:
cluster_strategy_map = {}

for cid, info in cluster_interpretation.items():
    cluster_strategy_map[cid] = assign_strategy(info)

cluster_strategy_map


{0: ['Flexible working hours and workload management'],
 1: ['Compensation review and financial incentives'],
 2: ['Compensation review and financial incentives'],
 3: ['Compensation review and financial incentives',
  'Flexible working hours and workload management']}

In [ ]:
df["Recommended_Retention_Strategies"] = df["Cluster"].map(cluster_strategy_map)


In [ ]:
df[["Cluster", "Recommended_Retention_Strategies"]].head(10)


,Cluster,Recommended_Retention_Strategies
0,1,[Compensation review and financial incentives]
1,3,"[Compensation review and financial incentives,..."
2,2,[Compensation review and financial incentives]
3,3,"[Compensation review and financial incentives,..."
4,2,[Compensation review and financial incentives]
5,2,[Compensation review and financial incentives]
6,2,[Compensation review and financial incentives]
7,2,[Compensation review and financial incentives]
8,3,"[Compensation review and financial incentives,..."
9,2,[Compensation review and financial incentives]
